## Report
```
The program generates a random DNA sequence of 10,000,000 nucleotides (A, C, G, T) represented as integers 0–3. A CUDA kernel maps each element to its complement using a lookup table, writing results to a parallel output array. The host launches the kernel across four block sizes — 64, 128, 256, and 512 — records GPU execution time with CUDA events, and prints the first 50 characters of both the original and complement strands. Block size 128 was the fastest at ~0.35 ms, while block size 64 was slower at ~0.57 ms due to lower occupancy. Block sizes 256 and 512 matched 128 closely, confirming that for a simple memory-bound lookup kernel.
```

In [10]:
%%writefile parallel_dna_complement_generation.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <time.h>

#define N 10000000

__constant__ char d_nucleotides[4] = {'A', 'C', 'G', 'T'};

__constant__ int d_complement[4] = {3, 2, 1, 0};

__global__ void parallel_dna_complement_generation_kernel(const int *input, int *output, int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n) {
        output[idx] = d_complement[input[idx]];
    }
}

void print_sequence(const char *label, const int *seq, int count)
{
    printf("%s: ", label);

    for (int i = 0; i < count; i++) {
        printf("%c", "ACGT"[seq[i]]);
    }

    printf("\n");
}

void launch_dna_kernel(const int *h_input, int block_size)
{
    printf("\nBlock size: %d\n", block_size);

    int *d_input = NULL;
    int *d_output = NULL;
    int *h_output = (int *)malloc(N * sizeof(int));

    cudaMalloc((void **)&d_input, N * sizeof(int));
    cudaMalloc((void **)&d_output, N * sizeof(int));

    cudaMemcpy(d_input, h_input, N * sizeof(int), cudaMemcpyHostToDevice);

    int grid_size = (N + block_size - 1) / block_size;
    long long total_threads = (long long)grid_size * block_size;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    parallel_dna_complement_generation_kernel<<<grid_size, block_size>>>(d_input, d_output, N);

    cudaEventRecord(stop);

    cudaEventSynchronize(stop);

    float ms = 0.0f;

    cudaEventElapsedTime(&ms, start, stop);

    cudaMemcpy(h_output, d_output, N * sizeof(int), cudaMemcpyDeviceToHost);

    print_sequence("Original", h_input, 50);
    print_sequence("Complement", h_output, 50);

    printf("Grid size: %d\n", grid_size);
    printf("Total threads launched: %lld\n", total_threads);
    printf("Kernel execution time: %.4f ms\n", ms);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_input);
    cudaFree(d_output);

    free(h_output);
}

int main(void)
{
    srand((unsigned int)time(NULL));

    int *h_input = (int *)malloc(N * sizeof(int));

    for (int i = 0; i < N; i++) {
        h_input[i] = rand() % 4;
    }

    int block_sizes[] = {64, 128, 256, 512};
    int num_configs = sizeof(block_sizes) / sizeof(block_sizes[0]);

    for (int i = 0; i < num_configs; i++) {
        launch_dna_kernel(h_input, block_sizes[i]);
    }

    free(h_input);

    return 0;
}

Overwriting parallel_dna_complement_generation.cu


In [11]:
!nvcc -arch=sm_75 parallel_dna_complement_generation.cu -o parallel_dna_complement_generation

In [12]:
!./parallel_dna_complement_generation


Block size: 64
Original: CGTTCTGAACTCACCTCCCTTGTTGGAGCAAGGAGATAATCTAGAGCCTT
Complement: GCAAGACTTGAGTGGAGGGAACAACCTCGTTCCTCTATTAGATCTCGGAA
Grid size: 156250
Total threads launched: 10000000
Kernel execution time: 0.5734 ms

Block size: 128
Original: CGTTCTGAACTCACCTCCCTTGTTGGAGCAAGGAGATAATCTAGAGCCTT
Complement: GCAAGACTTGAGTGGAGGGAACAACCTCGTTCCTCTATTAGATCTCGGAA
Grid size: 78125
Total threads launched: 10000000
Kernel execution time: 0.3467 ms

Block size: 256
Original: CGTTCTGAACTCACCTCCCTTGTTGGAGCAAGGAGATAATCTAGAGCCTT
Complement: GCAAGACTTGAGTGGAGGGAACAACCTCGTTCCTCTATTAGATCTCGGAA
Grid size: 39063
Total threads launched: 10000128
Kernel execution time: 0.3502 ms

Block size: 512
Original: CGTTCTGAACTCACCTCCCTTGTTGGAGCAAGGAGATAATCTAGAGCCTT
Complement: GCAAGACTTGAGTGGAGGGAACAACCTCGTTCCTCTATTAGATCTCGGAA
Grid size: 19532
Total threads launched: 10000384
Kernel execution time: 0.3728 ms
